# **시작이 제일 무서워 미룬이**

In [ ]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
from __future__ import annotations

from typing import TypedDict, List, Optional, Literal
from datetime import datetime, timedelta
import math

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langgraph.graph import StateGraph, START, END

## 1. 모델, 임베딩 설정
- OpenAI API 사용 (gpt-4.1-mini)

In [ ]:
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0.7,
)

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

## 2. 상태 타입 정의

In [ ]:
class GameState(TypedDict):
    # 입력값
    task: str
    feeling: str
    available_time: str
    deadline_text: str

    # 추론값
    inferred_state: str
    similarity_score: float
    urgency: str
    days_left: Optional[int]

    # 생성 결과
    quests: List[str]

    # 진행 상태
    current_step: int
    completed_steps: List[str]
    failed_steps: List[str]

    # 보상 / 요약
    reward: int
    session_summary: str

## 3. 내부 상태 후보

In [ ]:
STATE_CANDIDATES = {
    "direction_unclear": [
        "무엇부터 시작해야 할지 모르겠음",
        "일이 너무 많아 보여서 시작이 어려움",
        "막막해서 손을 못 대겠음",
        "어디서부터 해야 할지 감이 안 옴",
    ],
    "burden_high": [
        "잘해야 한다는 부담이 큼",
        "실수할까 봐 두려움",
        "완벽하게 해야 할 것 같아서 시작이 어려움",
        "압박감이 커서 손이 안 감",
    ],
    "low_energy": [
        "피곤해서 몸이 안 움직임",
        "기운이 없어서 시작하기 어려움",
        "졸리고 무기력함",
        "에너지가 너무 없음",
    ],
    "distracted": [
        "집중이 안 됨",
        "자꾸 딴짓하게 됨",
        "심리적 압박이 심함",
        "환경 때문에 흐트러짐",
    ],
    "avoidance": [
        "하기 싫어서 자꾸 미루게 됨",
        "그 일을 쳐다보고 싶지 않음",
        "회피하고 싶은 마음이 큼",
        "귀찮고 싫어서 도망가고 싶음",
    ],
}

STATE_LABELS_KO = {
    "direction_unclear": "시작점이 잘 보이지 않는 상태",
    "burden_high": "부담감이 큰 상태",
    "low_energy": "에너지가 낮은 상태",
    "distracted": "집중이 흐트러진 상태",
    "avoidance": "감정적으로 회피하고 있는 상태",
}


# 상태 후보 임베딩은 한 번만 미리 계산
STATE_EXAMPLE_VECTORS = {
    state_key: embeddings.embed_documents(examples)
    for state_key, examples in STATE_CANDIDATES.items()
}

## 4. 공통 함수

In [ ]:
def cosine_similarity(vec1: List[float], vec2: List[float]) -> float:
    dot = sum(a * b for a, b in zip(vec1, vec2))
    norm1 = math.sqrt(sum(a * a for a in vec1))
    norm2 = math.sqrt(sum(b * b for b in vec2))

    if norm1 == 0 or norm2 == 0:
        return 0.0

    return dot / (norm1 * norm2)


from datetime import datetime, timedelta

def parse_deadline(deadline_text: str) -> datetime | None:
    now = datetime.now()
    deadline_text = deadline_text.strip()

    if deadline_text.endswith("시간 뒤"):
        try:
            hours = int(deadline_text.replace("시간 뒤", "").strip())
            return now + timedelta(hours=hours)
        except ValueError:
            return None

    elif deadline_text == "오늘":
        return datetime.combine(now.date(), datetime.max.time())

    elif deadline_text == "내일":
        target_date = now.date() + timedelta(days=1)
        return datetime.combine(target_date, datetime.max.time())

    elif deadline_text.endswith("일 뒤"):
        try:
            days = int(deadline_text.replace("일 뒤", "").strip())
            target_date = now.date() + timedelta(days=days)
            return datetime.combine(target_date, datetime.max.time())
        except ValueError:
            return None

    else:
        try:
            return datetime.strptime(deadline_text, "%Y-%m-%d")
        except ValueError:
            return None


## 5. STT / TTS 포함
- 실제 구현 전에는 더미 함수로(?)

### 5-1. 입력 함수

In [ ]:
from typing import Literal

def stt_listen(prompt_text: str) -> str:
    print(f"[음성 입력 안내] {prompt_text}")
    return input(">>> ").strip()   # 지금은 테스트용


def get_user_input(prompt_text: str, mode: Literal["text", "voice"] = "text") -> str:
    if mode == "voice":
        return stt_listen(prompt_text)
    return input(prompt_text).strip()

### 5-2. 출력함수

In [ ]:
def tts_speak(text: str) -> None:
    print(f"[TTS] {text}")   # 지금은 테스트용


def output_message(text: str, mode: Literal["text", "voice", "both"] = "text") -> None:
    print(text)  # 항상 출력

    if mode in ("voice", "both"):
        tts_speak(text)

### 5-3. 입력 함수 모음

In [ ]:
def collect_input(input_mode: Literal["text", "voice"] = "text") -> dict:
    task = get_user_input("하기 싫은 일을 입력/말해주세요: ", input_mode)
    feeling = get_user_input("지금 상태를 입력/말해주세요: ", input_mode)
    available_time = get_user_input("가능한 시간을 입력/말해주세요: ", input_mode)
    deadline_text = get_user_input("데드라인을 입력/말해주세요: ", input_mode)

    return {
        "task": task,
        "feeling": feeling,
        "available_time": available_time,
        "deadline_text": deadline_text
    }

## 6. LangChain 기반 상태 추론

In [ ]:
def infer_state(feeling: str) -> dict:
    user_vec = embeddings.embed_query(feeling)

    best_state = None
    best_score = -1.0

    for state_key, vectors in STATE_EXAMPLE_VECTORS.items():
        for vec in vectors:
            score = cosine_similarity(user_vec, vec)
            if score > best_score:
                best_score = score
                best_state = state_key

    return {
        "inferred_state": STATE_LABELS_KO[best_state],
        "similarity_score": best_score,
    }

## 7. 퀘스트 생성 프롬프트

In [ ]:
import re

def parse_quest_lines(text: str) -> List[str]:
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    quests = []

    for line in lines:
        cleaned = re.sub(r"^\s*\d+[\.\)]\s*", "", line)   # 1. / 1) 제거
        cleaned = re.sub(r"^\s*[-*]\s*", "", cleaned)     # - / * 제거
        if cleaned:
            quests.append(cleaned)

    return quests[:5]

In [ ]:
SYSTEM_PROMPT = """
당신은 '시작이 제일 무서워 미룬이' 게임의 퀘스트 생성기입니다.

목표:
- 사용자가 하기 싫은 일을 지금 당장 시작할 수 있게 돕습니다.
- 완벽하게 끝내게 하는 것보다, 행동을 시작하게 하는 것이 우선입니다.

규칙:
- 반드시 5단계 퀘스트를 만듭니다.
- 각 단계는 지금 당장 실행 가능한 매우 구체적인 행동이어야 합니다.
- 추상적인 표현(예: 열심히 하기, 집중하기, 잘 해보기)은 금지합니다.
- 단계가 뒤로 갈수록 조금씩 더 진행되게 만듭니다.
- 마지막 5단계는 저장, 제출, 정리, 마무리 혹은 그와 비슷한 표현이어야 합니다.
- 사용자의 상태가 부담이 크거나 에너지가 낮으면 첫 단계는 매우 작게 만듭니다.
- 데드라인이 가까우면 필수 산출물 중심으로 구성합니다.
- 가능한 시간 안에 시도할 수 있는 수준으로 크기를 조절합니다.

출력 형식:
오직 5줄만 출력하세요.

1. ...
2. ...
3. ...
4. ...
5. ...
""".strip()


def generate_quests(state: GameState) -> List[str]:
    user_prompt = f"""
하기 싫은 일: {state['task']}
사용자 상태 입력: {state['feeling']}
추론된 상태: {state['inferred_state']}
가능한 시간: {state['available_time']}
데드라인: {state['deadline_text']}
긴급도: {state['urgency']}
남은 일수: {state['days_left']}
""".strip()

    response = llm.invoke([
        ("system", SYSTEM_PROMPT),
        ("user", user_prompt),
    ])

    text = response.content.strip()
    return parse_quest_lines(text)

## 8. 실패 시 더 작은 퀘스트 생성

In [ ]:
SMALLER_STEP_PROMPT = """
당신은 사용자가 이미 받은 퀘스트가 너무 부담스러울 때,
그 퀘스트를 더 작은 행동 하나로 다시 바꿔주는 도우미입니다.

규칙:
- 반드시 한 문장만 출력합니다.
- 지금 당장 할 수 있는 매우 작은 행동이어야 합니다.
- 추상적 표현은 금지합니다.
- 가능한 한 10초~1분 안에 시작할 수 있어야 합니다.
""".strip()


def retry_or_adjust(failed_quest: str, state: GameState) -> str:
    user_prompt = f"""
원래 퀘스트: {failed_quest}
사용자 상태: {state['inferred_state']}
가능 시간: {state['available_time']}
데드라인: {state['deadline_text']}
""".strip()

    response = llm.invoke([
        ("system", SMALLER_STEP_PROMPT),
        ("user", user_prompt),
    ])

    return response.content.strip()

## 9. 회고 생성

In [ ]:
SUMMARY_PROMPT = """
당신은 게임 세션 회고 도우미입니다.

규칙:
- 2~3문장으로 짧게 작성합니다.
- 비난하지 않습니다.
- 완료한 점을 먼저 짚고, 다음에 시도할 작은 행동 하나를 제안합니다.
""".strip()


def summarize_session_with_llm(state: GameState) -> str:
    user_prompt = f"""
하기 싫은 일: {state['task']}
완료한 단계: {state['completed_steps']}
실패하거나 조정한 단계: {state['failed_steps']}
총 보상: {state['reward']}
""".strip()

    response = llm.invoke([
        ("system", SUMMARY_PROMPT),
        ("user", user_prompt),
    ])

    return response.content.strip()

## 10. LangGraph 노드

In [ ]:
def infer_state_node(state: GameState) -> dict:
    return infer_state(state["feeling"])


def compute_urgency_node(state: GameState) -> dict:
    deadline_dt = parse_deadline(state["deadline_text"])
    now = datetime.now()

    if deadline_dt is None:
        return {
            "urgency": "긴급도 정보 없음",
            "days_left": None,
        }

    delta = deadline_dt - now
    hours_left = delta.total_seconds() / 3600

    if hours_left <= 24:
        urgency = "매우 높음"
    elif hours_left <= 72:
        urgency = "중간"
    else:
        urgency = "낮음"

    return {
        "urgency": urgency,
        "days_left": max(0, delta.days),
    }

def generate_quests_node(state: GameState) -> dict:
    quests = generate_quests(state)
    return {
        "quests": quests,
        "current_step": 0,
    }


## 11. LangGraph 그래프 구성
- 입력 -> 상태 추론 -> 긴급도 -> 퀘스트 생성

In [ ]:
graph_builder = StateGraph(GameState)

graph_builder.add_node("infer_state", infer_state_node)
graph_builder.add_node("compute_urgency", compute_urgency_node)
graph_builder.add_node("generate_quests", generate_quests_node)

graph_builder.add_edge(START, "infer_state")
graph_builder.add_edge("infer_state", "compute_urgency")
graph_builder.add_edge("compute_urgency", "generate_quests")
graph_builder.add_edge("generate_quests", END)

game_graph = graph_builder.compile()

## 12. 입력, 출력, 플레이 진행

In [ ]:
def collect_input(input_mode: Literal["text", "voice"] = "text") -> dict:
    task = get_user_input("하기 싫은 일을 입력/말해주세요: ", input_mode)
    feeling = get_user_input("지금 상태를 입력/말해주세요: ", input_mode)
    available_time = get_user_input("가능한 시간을 입력/말해주세요: ", input_mode)
    deadline_text = get_user_input("데드라인을 입력/말해주세요: ", input_mode)

    return {
        "task": task,
        "feeling": feeling,
        "available_time": available_time,
        "deadline_text": deadline_text,
    }


def present_quests(quests: List[str], output_mode: Literal["text", "voice", "both"] = "text") -> None:
    output_message("\n=== 생성된 퀘스트 ===", output_mode)
    for i, quest in enumerate(quests, start=1):
        output_message(f"{i}. {quest}", output_mode)


def play_steps(
    state: GameState,
    input_mode: Literal["text", "voice"] = "text",
    output_mode: Literal["text", "voice", "both"] = "text",
) -> GameState:
    while state["current_step"] < len(state["quests"]):
        step_idx = state["current_step"]
        quest = state["quests"][step_idx]

        output_message(f"\n현재 퀘스트: {quest}", output_mode)
        result = get_user_input("성공이면 1, 실패면 0을 입력/말해주세요: ", input_mode)

        if result == "1":
            state["completed_steps"].append(quest)
            gained = reward_for_step(step_idx + 1)
            state["reward"] += gained
            state["current_step"] += 1
            output_message(f"성공! {gained} 보상을 얻었습니다.", output_mode)
        else:
            state["failed_steps"].append(quest)
            smaller_quest = retry_or_adjust(quest, state)
            output_message(f"더 작은 퀘스트로 바꿉니다: {smaller_quest}", output_mode)
            state["quests"][step_idx] = smaller_quest

    return state

## 13. 전체 실행 함수

In [ ]:
def collect_input(input_mode: Literal["text", "voice"] = "text") -> dict:
    task = get_user_input("하기 싫은 일을 입력/말해주세요: ", input_mode)
    feeling = get_user_input("지금 상태를 입력/말해주세요: ", input_mode)
    available_time = get_user_input("가능한 시간을 입력/말해주세요: ", input_mode)
    deadline_text = get_user_input("데드라인을 입력/말해주세요: ", input_mode)

    return {
        "task": task,
        "feeling": feeling,
        "available_time": available_time,
        "deadline_text": deadline_text,
    }


def present_quests(quests: List[str], output_mode: Literal["text", "voice", "both"] = "text") -> None:
    output_message("\n=== 생성된 퀘스트 ===", output_mode)
    for i, quest in enumerate(quests, start=1):
        output_message(f"{i}. {quest}", output_mode)


def play_steps(
    state: GameState,
    input_mode: Literal["text", "voice"] = "text",
    output_mode: Literal["text", "voice", "both"] = "text",
) -> GameState:
    while state["current_step"] < len(state["quests"]):
        step_idx = state["current_step"]
        quest = state["quests"][step_idx]

        output_message(f"\n현재 퀘스트: {quest}", output_mode)
        result = get_user_input("성공이면 1, 실패면 0을 입력/말해주세요: ", input_mode)

        if result == "1":
            state["completed_steps"].append(quest)
            gained = reward_for_step(step_idx + 1)
            state["reward"] += gained
            state["current_step"] += 1
            output_message(f"성공! {gained} 보상을 얻었습니다.", output_mode)
        else:
            state["failed_steps"].append(quest)
            smaller_quest = retry_or_adjust(quest, state)
            output_message(f"더 작은 퀘스트로 바꿉니다: {smaller_quest}", output_mode)
            state["quests"][step_idx] = smaller_quest

    return state